# Proyecto de Grado: IA al servicio de las comunidades

Maestría en Analítica para la Inteligencia de Negocios

**Nombres:** María Fernanda Izquierdo Aparicio & Ana María Ochoa Muñoz

**Fecha:** 25 de Marzo de 2025

Este modelo forma parte del proyecto de grado de la Maestría en Analítica para la Inteligencia de Negocios de la Pontificia Universidad Javeriana, enfocado en la aplicación de Inteligencia Artificial al servicio de las comunidades. Su objetivo es clasificar imágenes de plantas en función de su especie. Se usa Random Forest el modelo. 

## 1. Importe de librerias

Se cargan las librerías necesarias para manipulación de datos, procesamiento de imágenes y modelado con redes neuronales. Algunas destacadas: (COMPLETAR)

In [ ]:
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import os
import zipfile
import shutil
import seaborn as sns
import datetime
import cv2


import keras
from keras import models
from keras import layers
from keras import optimizers

import random

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras import Layer
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import wrappers
from scikeras.wrappers import KerasClassifier, KerasRegressor

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestClassifier

import torch
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim

from PIL import Image  # Librería para abrir y manipular imágenes

# ignoring warnings
import warnings
warnings.simplefilter("ignore")

#import mglearn
from cycler import cycler


## 2. Importe de Datos
* Se importa un archivo CSV (Data_Final.csv) con el listado de imágenes y etiquetas.
* Se eliminan registros de la clase "Orange".
* Se divide el dataset en 80% entrenamiento - 20% prueba, asegurando balance entre clases con stratify.

In [ ]:
train=pd.read_csv('Data_Final.csv', sep=";")
train.head()

In [ ]:
#Quitar las naranjas 
train = train[train["Planta"] != "Orange"]
train = train[train["Planta"] != "Peach"]
train = train[train["Planta"] != "Grape"]

In [ ]:
# Rutas de tu carpeta y archivo CSV
ruta_carpeta_imagenes = 'Datos Finales'
ruta_archivo_csv = 'Data_Final.csv'

def cargar_imagenes_y_etiquetas(ruta_carpeta, ruta_csv):
    """
    Carga imágenes y etiquetas desde una carpeta y un archivo CSV.

    Args:
        ruta_carpeta: Ruta a la carpeta que contiene las imágenes.
        ruta_csv: Ruta al archivo CSV con las etiquetas.

    Returns:
        Tuplas con matrices de imagenes y etiquetas.
    """
    imagenes = []
    etiquetas = []
    df = pd.read_csv('Data_Final.csv', sep=";")
    for index, row in df.iterrows():
        nombre_archivo = row['Nombre del archivo']  # Asegúrate de que el nombre de la columna coincida
        etiqueta = row['Planta']  # Asegúrate de que el nombre de la columna coincida
        ruta_imagen = os.path.join(ruta_carpeta, nombre_archivo)
        imagen = cv2.imread(ruta_imagen)
        if imagen is not None:
            imagen_redimensionada = cv2.resize(imagen, (128, 128))
            imagen_aplanada = imagen_redimensionada.flatten()
            imagenes.append(imagen_aplanada)
            etiquetas.append(etiqueta)
        else:
            print(f"No se pudo cargar la imagen: {ruta_imagen}")

    return np.array(imagenes), np.array(etiquetas)

# Cargar imágenes y etiquetas
imagenes, etiquetas = cargar_imagenes_y_etiquetas(ruta_carpeta_imagenes, ruta_archivo_csv)

## 3. Análisis de Datos y Creación de conjuntos de train & test

* Se analiza la distribución de imágenes por categoría mediante gráficos de conteo.
* Se visualizan imágenes de plantas saludables y enfermas usando OpenCV.
* Se obtiene el tamaño de las imágenes para estandarizar el preprocesamiento.

### 3.1. Conteo de imágenes por categoría Plantas (Especies)

In [ ]:
train.Planta.value_counts()

In [ ]:
num_categorias = train['Planta'].nunique()
print(num_categorias)

### 3.2. Division aleatoria entre conjunto de train y test usando una dividion 80% - 20%

In [ ]:
# Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(imagenes, etiquetas, test_size=0.2, random_state=42)

### 3.3. Conteo de imágenes por en el conjunto train y test por categoría

In [ ]:
X_train.size

In [ ]:
X_test.size

## 4. Preparación del Modelo

* Se implementará data augmentation para mejorar la generalización.
* Se evaluará el modelo con métricas como precisión y recall

In [ ]:
#Generación del Random Forest
rf = RandomForestClassifier(n_estimators = num_categorias, random_state=2) # Numero de estimadores: Cantida de árboles,
rf.fit(X_train, y_train)

In [ ]:
# Acceder a los árboles individuales
Arbol_1 = rf.estimators_[0]  # Accede al primer árbol
Arbol_2 = rf.estimators_[1]  # Accede al segundo árbol
Arbol_3 = rf.estimators_[2]
Arbol_4 = rf.estimators_[3]
Arbol_5 = rf.estimators_[4]
Arbol_6 = rf.estimators_[5]
Arbol_7 = rf.estimators_[6]
Arbol_8 = rf.estimators_[7]
Arbol_9 = rf.estimators_[8]

# Ahora puedes examinar los detalles de cada árbol individualmente, por ejemplo, la profundidad del árbol
Arbol_1_depth = Arbol_1.tree_.max_depth
Arbol_2_depth = Arbol_2.tree_.max_depth
Arbol_3_depth = Arbol_3.tree_.max_depth
Arbol_4_depth = Arbol_4.tree_.max_depth
Arbol_5_depth = Arbol_5.tree_.max_depth
Arbol_6_depth = Arbol_6.tree_.max_depth
Arbol_7_depth = Arbol_7.tree_.max_depth
Arbol_8_depth = Arbol_8.tree_.max_depth
Arbol_9_depth = Arbol_9.tree_.max_depth


print("Profundidad del primer árbol:", Arbol_1_depth)
print("Profundidad del segundo árbol:", Arbol_2_depth)
print("Profundidad del tercer árbol:", Arbol_3_depth)
print("Profundidad del cuarto árbol:", Arbol_4_depth)
print("Profundidad del quinto árbol:", Arbol_5_depth)
print("Profundidad del sexto árbol:", Arbol_6_depth)
print("Profundidad del septimo árbol:", Arbol_7_depth)
print("Profundidad del octavo árbol:", Arbol_8_depth)
print("Profundidad del noveno árbol:", Arbol_9_depth)

In [ ]:
rf.estimators_